# Comparación Donut v2 vs v3 — Tickets de Mercadona

Evalúa los dos modelos **sobre los mismos 14 tickets de validación** y con **las mismas métricas**,
midiendo lo que de verdad ve el usuario en la aplicación: cada modelo pasa por su propio post-procesado
del backend (v2 con el parser defensivo y la cantidad forzada a 1; v3 con su formato de tokens).

**Archivos necesarios** (súbelos a `/content` o déjalos en una carpeta de Google Drive):

| Archivo | Contenido |
|---|---|
| `mi_modelo_mercadona_v2.zip` | modelo antiguo (en tu PC se llama `mi_modelo_mercadona_v2 .zip`, con un espacio: da igual) |
| `mi_modelo_mercadona_v3.zip` | modelo nuevo, el que genera el cuaderno de entrenamiento |
| `datos_donut_v3.zip` | los PDF y scripts, para reconstruir exactamente el mismo conjunto de validación |

Con GPU T4 basta. Los modelos se cargan **de uno en uno** para no llenar la memoria.

> ⚠️ El modelo v2 se entrenó con los 91 tickets, así que **ya había visto** los 14 de validación: su resultado
> es optimista. Aun así, la comparación es justa en el sentido de que favorece al modelo antiguo.

In [ ]:
!pip install -q -U transformers sentencepiece pdfplumber
!pip uninstall -y -q torchaudio   # evita el conflicto de versiones CUDA de Colab

# ── Dónde están los zips ──
# Opción A: súbelos a /content con el panel de archivos (izquierda).
# Opción B (recomendada para ~1 GB): ponlos en una carpeta de Drive y descomenta estas dos líneas.
# from google.colab import drive; drive.mount('/content/drive')
# CARPETA_ZIPS = "/content/drive/MyDrive/TFG"
CARPETA_ZIPS = "/content"

## 1. Descomprimir y reconstruir el conjunto de validación

In [ ]:
import glob, os, shutil, subprocess
from pathlib import Path

def buscar_zip(patron):
    candidatos = sorted(glob.glob(os.path.join(CARPETA_ZIPS, patron)))
    assert candidatos, f"No encuentro {patron} en {CARPETA_ZIPS}"
    return candidatos[0]

def descomprimir_modelo(zip_path, destino):
    shutil.rmtree(destino, ignore_errors=True)
    subprocess.run(["unzip", "-q", "-o", zip_path, "-d", destino], check=True)
    # La carpeta del modelo es la que contiene config.json, esté donde esté dentro del zip
    config = sorted(Path(destino).rglob("config.json"), key=lambda p: len(p.parts))[0]
    return str(config.parent)

RUTA_V2 = descomprimir_modelo(buscar_zip("mi_modelo_mercadona_v2*.zip"), "/content/modelo_v2")
RUTA_V3 = descomprimir_modelo(buscar_zip("mi_modelo_mercadona_v3*.zip"), "/content/modelo_v3")
subprocess.run(["unzip", "-q", "-o", buscar_zip("datos_donut_v3*.zip"), "-d", "/content/"], check=True)
!python /content/datos_donut_v3/preparar_dataset.py /content/datos_donut_v3/pdfs /content/datos_donut_v3/dataset_v3
print("v2:", RUTA_V2, "\nv3:", RUTA_V3)

## 2. Ground Truth y métricas comunes
Las dos salidas se convierten al formato que guarda la aplicación: lista de
`(descripción, cantidad, precio unitario)`, fecha y total. Es una comparación **del sistema completo**
(modelo + post-procesado del backend de cada versión).

Como el backend v2 forzaba la cantidad a 1, en los productos con varias unidades nunca podía acertar.
Para aislar la capacidad de **lectura** del modelo se añade también el *F1 solo de nombres de producto*,
que ignora cantidades y precios.

In [ ]:
import json, re, time, gc
from collections import Counter
import torch
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel

DATASET = Path("/content/datos_donut_v3/dataset_v3")
device = "cuda" if torch.cuda.is_available() else "cpu"

validacion = []
for linea in open(DATASET / "validation" / "metadata.jsonl", encoding="utf-8"):
    r = json.loads(linea)
    validacion.append({"imagen": str(DATASET / "validation" / r["file_name"]),
                       "origen": r.get("origen", ""),
                       "gt": json.loads(r["ground_truth"])["gt_parse"]})
print(len(validacion), "tickets de validación,",
      sum(len(d["gt"]["menu"]) for d in validacion), "productos")

def euros(txt):
    txt = str(txt or "").replace("€", "").replace("/kg", "").strip()
    txt = txt.replace(".", "").replace(",", ".")
    try:
        return round(float(txt), 2)
    except ValueError:
        return None

def gt_a_app(gt):
    # Ground Truth -> lo que debería guardar la aplicación
    items = []
    for it in gt["menu"]:
        cnt = int(it["cnt"])
        if "weight" in it:
            items.append((it["nm"], 1, euros(it["price"])))
        elif "unitprice" in it:
            items.append((it["nm"], cnt, euros(it["unitprice"])))
        else:
            items.append((it["nm"], cnt, round(euros(it["price"]) / cnt, 2) if cnt else euros(it["price"])))
    return {"fecha": gt["date"], "total": euros(gt["total"]["total_price"]), "items": items}

def comparar(pred, gt):
    p, g = Counter(pred["items"]), Counter(gt["items"])
    tp = sum((p & g).values())
    pn, gn = Counter(i[0] for i in pred["items"]), Counter(i[0] for i in gt["items"])
    tp_nombres = sum((pn & gn).values())
    errores = max(sum((p - g).values()), sum((g - p).values()))
    fecha_ok = str(pred["fecha"] or "")[:10] == gt["fecha"]
    total_ok = pred["total"] == gt["total"]
    return {"tp": tp, "tp_nombres": tp_nombres, "n_pred": sum(p.values()), "n_gt": sum(g.values()), "errores": errores,
            "fecha_ok": fecha_ok, "total_ok": total_ok,
            "perfecto": fecha_ok and total_ok and errores == 0,
            "casi_perfecto": fecha_ok and total_ok and errores <= 2}

def resumen(filas):
    n = len(filas)
    tp = sum(f["tp"] for f in filas)
    prec = tp / max(1, sum(f["n_pred"] for f in filas))
    rec = tp / max(1, sum(f["n_gt"] for f in filas))
    tp_n = sum(f["tp_nombres"] for f in filas)
    return {
        "Tickets perfectos (%)": 100 * sum(f["perfecto"] for f in filas) / n,
        "Tickets casi perfectos, <=2 fallos (%)": 100 * sum(f["casi_perfecto"] for f in filas) / n,
        "F1 productos (%)": 100 * (2 * prec * rec / max(1e-9, prec + rec)),
        "F1 solo nombres de producto (%)": 100 * 2 * tp_n / max(1, sum(f["n_pred"] + f["n_gt"] for f in filas)),
        "Precisión productos (%)": 100 * prec,
        "Recall productos (%)": 100 * rec,
        "Acierto del total (%)": 100 * sum(f["total_ok"] for f in filas) / n,
        "Acierto de la fecha (%)": 100 * sum(f["fecha_ok"] for f in filas) / n,
        "Salida estructurada válida (%)": 100 * sum(f["salida_valida"] for f in filas) / n,
        "Tiempo medio por ticket (s)": sum(f["segundos"] for f in filas) / n,
    }

def token2json(tokens, interno=False):
    # Adaptado del repositorio oficial de Donut (clovaai/donut, licencia MIT)
    salida = {}
    while tokens:
        inicio = re.search(r"<s_(.*?)>", tokens, re.IGNORECASE)
        if inicio is None:
            break
        clave = inicio.group(1)
        fin = re.search(rf"</s_{re.escape(clave)}>", tokens, re.IGNORECASE)
        inicio = inicio.group()
        if fin is None:
            tokens = tokens.replace(inicio, "")
            continue
        fin = fin.group()
        contenido = re.search(f"{re.escape(inicio)}(.*?){re.escape(fin)}", tokens, re.IGNORECASE | re.DOTALL)
        if contenido is not None:
            contenido = contenido.group(1).strip()
            if "<s_" in contenido and "</s_" in contenido:
                valor = token2json(contenido, interno=True)
                if valor:
                    salida[clave] = valor[0] if len(valor) == 1 else valor
            else:
                hojas = [h.strip() for h in contenido.split("<sep/>")]
                salida[clave] = hojas[0] if len(hojas) == 1 else hojas
        tokens = tokens[tokens.find(fin) + len(fin):].strip()
        if tokens[:6] == "<sep/>":
            return [salida] + token2json(tokens[6:], interno=True)
    if salida:
        return [salida] if interno else salida
    return [] if interno else {"text_sequence": tokens}

def liberar(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()

## 3. Modelo v2 (tal y como funcionaba en el backend antiguo)
Mismos parámetros de generación (`repetition_penalty=1.15`) y mismo post-procesado: `json.loads` y, si falla,
el parser defensivo con expresiones regulares. La cantidad se fuerza a 1, como hacía el backend.

In [ ]:
def rescatar_json_roto(texto):
    m = re.search(r'[\'"]date[\'"]:\s*[\'"]([^\'"]+)[\'"]', texto)
    fecha = m.group(1) if m else None
    m = re.search(r'[\'"]total_price[\'"]:\s*[\'"]([^\'"]+)[\'"]', texto)
    total = euros(m.group(1)) if m else 0.0
    items = []
    for m in re.finditer(r'[\'"]nm[\'"]:\s*[\'"]([^\'"]+)[\'"].*?(?:[\'"]cnt[\'"]|[\'"]price[\'"]):\s*[\'"]([\d,\.\-]+)[\'"]', texto):
        if len(m.group(1).strip()) < 3:
            continue
        items.append((m.group(1).strip(), 1, euros(m.group(2))))
    return fecha, total, items

def postprocesado_v2(texto):
    valido = False
    datos = {}
    if "{" in texto:
        try:
            datos = json.loads(texto[texto.find("{"):])
            datos = datos.get("gt_parse", datos)
            valido = isinstance(datos, dict) and bool(datos.get("menu"))
        except json.JSONDecodeError:
            pass
    if valido:
        menu = datos.get("menu", [])
        menu = [menu] if isinstance(menu, dict) else menu
        items = [(str(x.get("nm", "")), 1, euros(x.get("price"))) for x in menu if isinstance(x, dict)]
        fecha = (datos.get("store_info") or {}).get("date")
        total = euros((datos.get("total") or {}).get("total_price"))
    else:
        fecha, total, items = rescatar_json_roto(texto)
    if not total and items:
        total = round(sum(i[2] or 0 for i in items), 2)
    return {"fecha": fecha, "total": total, "items": items}, valido

proc = DonutProcessor.from_pretrained(RUTA_V2)
modelo = VisionEncoderDecoderModel.from_pretrained(RUTA_V2).to(device).eval()
tok = proc.tokenizer
prompt = tok("<s_cord-v2>", add_special_tokens=False, return_tensors="pt").input_ids.to(device)
print("Resolución v2:", proc.image_processor.size)

filas_v2 = []
for d in validacion:
    img = Image.open(d["imagen"]).convert("RGB")
    t0 = time.time()
    pv = proc(images=img, return_tensors="pt").pixel_values.to(device)
    with torch.no_grad():
        out = modelo.generate(pv, decoder_input_ids=prompt, max_length=modelo.decoder.config.max_position_embeddings,
                              repetition_penalty=1.15, num_beams=1, use_cache=True,
                              pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id,
                              bad_words_ids=[[tok.unk_token_id]])
    texto = proc.batch_decode(out)[0].replace(tok.eos_token, "").replace(tok.pad_token, "").replace("<s_cord-v2>", "")
    pred, valido = postprocesado_v2(texto)
    seg = time.time() - t0
    fila = comparar(pred, gt_a_app(d["gt"]))
    fila.update({"imagen": Path(d["imagen"]).name, "salida_valida": valido, "segundos": seg, "salida": texto})
    filas_v2.append(fila)
    print(f"{fila['imagen']}: {fila['tp']}/{fila['n_gt']} productos, errores={fila['errores']}, "
          f"total={'✓' if fila['total_ok'] else '✗'}, fecha={'✓' if fila['fecha_ok'] else '✗'}, JSON válido={valido}, {seg:.1f}s")

del modelo, proc; liberar()

## 4. Modelo v3

In [ ]:
proc = DonutProcessor.from_pretrained(RUTA_V3)
modelo = VisionEncoderDecoderModel.from_pretrained(RUTA_V3).to(device).eval()
tok = proc.tokenizer
cfg = json.load(open(f"{RUTA_V3}/donut_mercadona.json", encoding="utf-8"))
prompt = tok(cfg["task_token"], add_special_tokens=False, return_tensors="pt").input_ids.to(device)
print("Resolución v3:", proc.image_processor.size)

def postprocesado_v3(texto):
    datos = token2json(texto)
    menu = datos.get("menu", []) if isinstance(datos, dict) else []
    menu = [menu] if isinstance(menu, dict) else [x for x in menu if isinstance(x, dict)]
    valido = bool(menu) and "total" in datos
    items = []
    for it in menu:
        importe = euros(it.get("price"))
        try:
            cnt = int(str(it.get("cnt", "1")).strip())
        except ValueError:
            cnt = 1
        if "weight" in it or cnt == 0:
            items.append((it.get("nm", ""), 1, importe))
        elif "unitprice" in it:
            items.append((it.get("nm", ""), cnt, euros(it["unitprice"])))
        else:
            items.append((it.get("nm", ""), cnt, round(importe / cnt, 2) if importe is not None else None))
    total = euros((datos.get("total") or {}).get("total_price")) if isinstance(datos.get("total"), dict) else None
    return {"fecha": datos.get("date"), "total": total, "items": items}, valido

filas_v3 = []
for d in validacion:
    img = Image.open(d["imagen"]).convert("RGB")
    t0 = time.time()
    pv = proc(images=img, return_tensors="pt").pixel_values.to(device)
    with torch.no_grad():
        out = modelo.generate(pv, decoder_input_ids=prompt, max_length=modelo.decoder.config.max_position_embeddings,
                              num_beams=1, use_cache=True, pad_token_id=tok.pad_token_id,
                              eos_token_id=tok.eos_token_id, bad_words_ids=[[tok.unk_token_id]])
    texto = proc.batch_decode(out)[0].replace(tok.eos_token, "").replace(tok.pad_token, "")
    texto = texto.replace(cfg["task_token"], "").translate(str.maketrans(cfg.get("mapeo_inverso", {})))
    pred, valido = postprocesado_v3(texto)
    seg = time.time() - t0
    fila = comparar(pred, gt_a_app(d["gt"]))
    fila.update({"imagen": Path(d["imagen"]).name, "salida_valida": valido, "segundos": seg, "salida": texto})
    filas_v3.append(fila)
    print(f"{fila['imagen']}: {fila['tp']}/{fila['n_gt']} productos, errores={fila['errores']}, "
          f"total={'✓' if fila['total_ok'] else '✗'}, fecha={'✓' if fila['fecha_ok'] else '✗'}, {seg:.1f}s")

del modelo, proc; liberar()

## 5. Comparación y exportación para la memoria

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

r2, r3 = resumen(filas_v2), resumen(filas_v3)
tabla = pd.DataFrame({"Donut v2": r2, "Donut v3": r3}).round(1)
tabla["Diferencia"] = (tabla["Donut v3"] - tabla["Donut v2"]).round(1)
display(tabla)

por_ticket = pd.DataFrame({
    "ticket": [f["imagen"] for f in filas_v3],
    "productos": [f["n_gt"] for f in filas_v3],
    "v2 aciertos": [f["tp"] for f in filas_v2],
    "v2 perfecto": [f["perfecto"] for f in filas_v2],
    "v3 aciertos": [f["tp"] for f in filas_v3],
    "v3 perfecto": [f["perfecto"] for f in filas_v3],
})
display(por_ticket)

# Gráfico de barras con las métricas en %
metricas = [k for k in r2 if k.endswith("(%)")]
etiquetas = [m.replace(" (%)", "") for m in metricas]
x = range(len(metricas))
fig, ax = plt.subplots(figsize=(11, 4.5))
b2 = ax.bar([i - 0.2 for i in x], [r2[m] for m in metricas], 0.4, label="Donut v2", color="#9aa5b1")
b3 = ax.bar([i + 0.2 for i in x], [r3[m] for m in metricas], 0.4, label="Donut v3", color="#2563eb")
ax.bar_label(b2, fmt="%.0f", fontsize=8); ax.bar_label(b3, fmt="%.0f", fontsize=8)
ax.set_xticks(list(x)); ax.set_xticklabels(etiquetas, rotation=20, ha="right", fontsize=9)
ax.set_ylim(0, 110); ax.set_ylabel("%"); ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"Validación: {len(validacion)} tickets, {sum(f['n_gt'] for f in filas_v3)} productos")
plt.tight_layout()
plt.savefig("/content/comparacion_v2_v3.png", dpi=200)
plt.show()

tabla.to_csv("/content/comparacion_v2_v3.csv")
por_ticket.to_csv("/content/comparacion_por_ticket.csv", index=False)
json.dump({"v2": filas_v2, "v3": filas_v3}, open("/content/comparacion_detalle.json", "w"), ensure_ascii=False, indent=2)
print(tabla.to_latex(caption="Comparación de Donut v2 y v3 en el conjunto de validación",
                     label="tab:comparacion-v2-v3", float_format="%.1f"))

In [ ]:
!cd /content && zip -q comparacion_v2_v3.zip comparacion_v2_v3.png comparacion_v2_v3.csv comparacion_por_ticket.csv comparacion_detalle.json
from google.colab import files
files.download("/content/comparacion_v2_v3.zip")